In [ ]:
# !pip install transformers

In [ ]:
# import torch
# print(torch.__version__)

In [ ]:
# import transformers
# print(transformers.__version__)

In [ ]:

# !pip install transformers==4.51.3

In [ ]:
# import torch, transformers

# print(torch.__version__)
# print(transformers.__version__)

In [ ]:
# import transformers
# print(transformers.__version__)

In [ ]:
!pip install trl

In [ ]:
pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 108.9 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


Now that `trl` is installed, please re-run the `SFTTrainer` definition cell (`64cc9730`), followed by the gradient checkpointing cell (`TherTEkotdAz`), and then the training cell (`HlxbKzD7r-nD`).

Now that `transformers` is updated, let's re-run the `SFTTrainer` definition to ensure all configurations are loaded correctly before attempting to train again.

In [ ]:
# Trainer

from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir="BalKatha-LoRA",
    num_train_epochs=2,
    learning_rate=2e-4,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    logging_steps=10,
    save_strategy="epoch",

    fp16=True,
    bf16=False,

    optim="adamw_torch", # Keeping this for now, will revisit if error persists

    report_to="none",

    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset,
    processing_class=tokenizer,
)

Tokenizing train dataset:   0%|          | 0/48 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/48 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/48 [00:00<?, ? examples/s]

In [ ]:
from datasets import load_dataset

In [ ]:
dataset = load_dataset("roneneldan/TinyStories")

In [ ]:
stories = dataset["train"].select(range(1000))

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM
)

In [ ]:
translator_name = "facebook/nllb-200-distilled-600M"

translator_tokenizer = AutoTokenizer.from_pretrained(
    translator_name
)

translator_model = AutoModelForSeq2SeqLM.from_pretrained(
    translator_name,
    device_map="auto"
)

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

In [ ]:
# translator_name = "facebook/nllb-200-1.3B"

# translator_tokenizer = AutoTokenizer.from_pretrained(translator_name)
# translator_model = AutoModelForSeq2SeqLM.from_pretrained(
#     translator_name,
#     device_map="auto",
#     torch_dtype=torch.float16 # Add this to cut VRAM usage in half
# )

In [ ]:
def translate_batch(texts):

    translator_tokenizer.src_lang = "eng_Latn"

    inputs = translator_tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    ).to(translator_model.device)

    generated = translator_model.generate(

        **inputs,

        forced_bos_token_id=translator_tokenizer.convert_tokens_to_ids(
            "hin_Deva"
        ),

        max_new_tokens=512

    )

    return translator_tokenizer.batch_decode(
        generated,
        skip_special_tokens=True
    )

In [ ]:
BATCH_SIZE = 16

english = []
hindi = []

In [ ]:
import tqdm

In [ ]:
for i in tqdm.tqdm(range(0, len(stories), BATCH_SIZE)):

    batch = stories.select(
        range(i, min(i+BATCH_SIZE, len(stories)))
    )

    # Explicitly convert the 'text' column to a list of strings
    en_batch = list(batch["text"])

    hi_batch = translate_batch(en_batch)

    english.extend(en_batch)

    hindi.extend(hi_batch)

  5%|▍         | 3/63 [00:16<05:26,  5.44s/it]


KeyboardInterrupt: 

In [ ]:
import pandas as pd

df = pd.DataFrame({

    "english": english,

    "hindi": hindi

})

df.to_csv(
    "TinyStoriesHindi.csv",
    index=False
)

In [ ]:
import pandas as pd

df = pd.read_csv("TinyStoriesHindi.csv")

In [ ]:
df.head()

,english,hindi
0,"One day, a little girl named Lily found a need...",एक दिन लिली नाम की एक छोटी लड़की ने अपने कमरे ...
1,"Once upon a time, there was a little car named...",एक समय था जब बीप नाम की एक छोटी कार थी. बीप को...
2,"One day, a little fish named Fin was swimming ...",एक दिन फिन नामक एक छोटी सी मछली तट के पास तैर ...
3,"Once upon a time, in a land full of trees, the...",एक बार एक बार पेड़ों से भरे एक देश में एक छोटा...
4,"Once upon a time, there was a little girl name...",एक समय था जब लिली नाम की एक छोटी लड़की थी। लिल...


In [ ]:
df.shape

(1000, 2)

In [ ]:
# Setting up the themes

themes = [

"Honesty",

"Kindness",

"Friendship",

"Helping Others",

"Teamwork",

"Courage",

"Responsibility",

"Patience",

"Curiosity",

"Sharing"

]

In [ ]:
# age groups

ages = [

"3-5",

"5-7",

"7-9"

]

In [ ]:
# prompt templates

templates = [

"Write a Hindi children's story for children aged {age} about {theme}.",

"Write a Hindi bedtime story teaching {theme} for children aged {age}.",

"Write a moral Hindi story on {theme} for kids aged {age}.",

"Generate a Hindi children's story focusing on {theme}.",

"Write a fun Hindi story with a moral about {theme}."

]

In [ ]:
import random

instructions = []

for _ in range(len(hindi)):

    template = random.choice(templates)

    theme = random.choice(themes)

    age = random.choice(ages)

    instructions.append(

        template.format(

            age=age,

            theme=theme

        )

    )

In [ ]:
import json
import torch

In [ ]:
instruction_dataset = []

for prompt, story in zip(
    instructions,
    hindi
):

    instruction_dataset.append({

        "messages":[

            {

                "role":"user",

                "content":prompt

            },

            {

                "role":"assistant",

                "content":story

            }

        ]

    })

In [ ]:
with open(
    "balkatha_instruction.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(

        instruction_dataset,

        f,

        ensure_ascii=False,

        indent=2

    )

In [ ]:
from datasets import Dataset

dataset = Dataset.from_list(
    instruction_dataset
)

dataset

Dataset({
    features: ['messages'],
    num_rows: 48
})

In [ ]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)


In [ ]:
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.float16,
#     bnb_4bit_use_double_quant=True,
# )

In [ ]:
# Loading Qwen

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [ ]:
model.config.use_cache = False

In [ ]:
# model = prepare_model_for_kbit_training(model)

In [ ]:
model.config.use_cache = False

model.gradient_checkpointing_enable()

model.enable_input_require_grads()

In [ ]:
# LoRA config

lora_config = LoraConfig(

    r=16,

    lora_alpha=32,

    lora_dropout=0.05,

    bias="none",

    task_type="CAUSAL_LM",

    target_modules=[

        "q_proj",

        "k_proj",

        "v_proj",

        "o_proj",

        "gate_proj",

        "up_proj",

        "down_proj",

    ],

)

In [ ]:
model = get_peft_model(model, lora_config)

In [ ]:
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

print("Trainable:", trainable)
print("Total:", total)

Trainable: 18464768
Total: 1562179072


In [ ]:
print(dataset[0])

{'messages': [{'role': 'user', 'content': "Write a Hindi children's story for children aged 7-9 about Honesty."}, {'role': 'assistant', 'content': 'एक दिन लिली नाम की एक छोटी लड़की ने अपने कमरे में एक सुई पाई। वह जानती थी कि इसके साथ खेलना मुश्किल है क्योंकि यह तेज था। लिली अपनी मां के साथ सुई साझा करना चाहती थी, इसलिए वह अपनी शर्ट पर एक बटन सीढ़ी सकती थी। लिली अपनी मां के पास गई और बोली, "माँ, मैंने यह सुई पाई है। क्या आप इसे मेरे साथ साझा कर सकते हैं और मेरी शर्ट सीढ़ी सकते हैं?" उसकी मां मुस्कुराई और बोली, "हाँ, लिली, हम सुई साझा कर सकते हैं और अपनी शर्ट ठीक कर सकते हैं। " उन्होंने एक साथ सुई साझा की और लिली की शर्ट पर बटन सीढ़ी। उनके लिए यह मुश्किल नहीं था क्योंकि वे साझा कर रहे थे और एक-दूसरे की मदद कर रहे थे। जब वे खत्म कर चुके थे, तो लिली ने अपनी मां को सुई साझा करने और अपनी शर्ट को ठीक करने के लिए धन्यवाद दिया। वे दोनों खुश महसूस करते थे क्योंकि वे साझा कर रहे थे और साथ काम कर रहे थे।'}]}


In [ ]:
# converting to chat

def format_chat(example):

    text = tokenizer.apply_chat_template(

        example["messages"],

        tokenize=False,

        add_generation_prompt=False,

    )

    return {

        "text": text

    }

dataset = dataset.map(format_chat)

Map:   0%|          | 0/48 [00:00<?, ? examples/s]

In [ ]:
print(dataset[0]["text"])

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Write a Hindi children's story for children aged 7-9 about Honesty.<|im_end|>
<|im_start|>assistant
एक दिन लिली नाम की एक छोटी लड़की ने अपने कमरे में एक सुई पाई। वह जानती थी कि इसके साथ खेलना मुश्किल है क्योंकि यह तेज था। लिली अपनी मां के साथ सुई साझा करना चाहती थी, इसलिए वह अपनी शर्ट पर एक बटन सीढ़ी सकती थी। लिली अपनी मां के पास गई और बोली, "माँ, मैंने यह सुई पाई है। क्या आप इसे मेरे साथ साझा कर सकते हैं और मेरी शर्ट सीढ़ी सकते हैं?" उसकी मां मुस्कुराई और बोली, "हाँ, लिली, हम सुई साझा कर सकते हैं और अपनी शर्ट ठीक कर सकते हैं। " उन्होंने एक साथ सुई साझा की और लिली की शर्ट पर बटन सीढ़ी। उनके लिए यह मुश्किल नहीं था क्योंकि वे साझा कर रहे थे और एक-दूसरे की मदद कर रहे थे। जब वे खत्म कर चुके थे, तो लिली ने अपनी मां को सुई साझा करने और अपनी शर्ट को ठीक करने के लिए धन्यवाद दिया। वे दोनों खुश महसूस करते थे क्योंकि वे साझा कर रहे थे और साथ काम कर रहे थे।<|im_end|>



In [ ]:
# Training Arguments

training_args = TrainingArguments(

    output_dir="BalKatha-LoRA",

    num_train_epochs=2,

    learning_rate=2e-4,

    per_device_train_batch_size=2,

    gradient_accumulation_steps=4,

    logging_steps=10,

    save_strategy="epoch",

    fp16=True,

    optim="paged_adamw_8bit",

    report_to="none",

    remove_unused_columns=False,

)

In [ ]:
import trl
print(trl.__version__)

1.8.0


In [ ]:
model.print_trainable_parameters()

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [ ]:
# Trainer

from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir="BalKatha-LoRA",
    num_train_epochs=2,
    learning_rate=2e-4,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    logging_steps=10,
    save_strategy="epoch",

    fp16=True,
    bf16=False,

    optim="adamw_torch",

    report_to="none",

    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset,
    processing_class=tokenizer,
)

Tokenizing train dataset:   0%|          | 0/48 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/48 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/48 [00:00<?, ? examples/s]

In [ ]:
# model.gradient_checkpointing_enable()

# model.enable_input_require_grads()

In [ ]:
batch = tokenizer(
    dataset[0]["text"],
    return_tensors="pt",
    truncation=True,
    max_length=256,
).to(model.device)

loss = model(**batch, labels=batch["input_ids"]).loss

print("Loss:", loss.item())

count = 0
for n, p in model.named_parameters():
    if p.requires_grad and p.grad is not None:
        count += 1

print("Parameters with gradients:", count)

Loss: 1.6899253129959106
Parameters with gradients: 0


In [ ]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Step,Training Loss
10,0.915002


TrainOutput(global_step=12, training_loss=0.895381768544515, metrics={'train_runtime': 27.2069, 'train_samples_per_second': 3.529, 'train_steps_per_second': 0.441, 'total_flos': 615885346111488.0, 'train_loss': 0.895381768544515, 'entropy': 0.8384294509887695, 'num_tokens': 68106.0, 'mean_token_accuracy': 0.7675446048378944, 'epoch': 2.0})

In [66]:
# Save

trainer.save_model("BalKatha-LoRA")

tokenizer.save_pretrained("BalKatha-LoRA")

('BalKatha-LoRA/tokenizer_config.json',
 'BalKatha-LoRA/chat_template.jinja',
 'BalKatha-LoRA/tokenizer.json')

In [68]:
# Merge LoRA

merged_model = model.merge_and_unload()

In [69]:
# Saving the final Model

merged_model.save_pretrained("BalKatha-Final")

tokenizer.save_pretrained("BalKatha-Final")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('BalKatha-Final/tokenizer_config.json',
 'BalKatha-Final/chat_template.jinja',
 'BalKatha-Final/tokenizer.json')

In [70]:
# Loading the fine tuned model

from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16
)

model = PeftModel.from_pretrained(
    base_model,
    "BalKatha-LoRA"
)

tokenizer = AutoTokenizer.from_pretrained("BalKatha-LoRA")

model.eval()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(

In [96]:
# # Generating story

# def generate_story(prompt):

#     messages = [
#         {
#             "role": "user",
#             "content": prompt
#         }
#     ]

#     text = tokenizer.apply_chat_template(
#         messages,
#         tokenize=False,
#         add_generation_prompt=True
#     )

#     inputs = tokenizer(
#         text,
#         return_tensors="pt"
#     ).to(model.device)

#     outputs = model.generate(
#         **inputs,
#         max_new_tokens=350,
#         temperature=0.8,
#         top_p=0.9,
#         do_sample=True,
#         repetition_penalty=1.1
#     )

#     generated = tokenizer.decode(
#         outputs[0],
#         skip_special_tokens=True
#     )

#     return generated

In [97]:
import torch

def generate_story(prompt):

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():

        outputs = model.generate(
            **inputs,
            max_new_tokens=256,          # reduced
            do_sample=False,             # much faster
            temperature=0.7,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=True
        )

    generated = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return generated

In [98]:
prompt = """
Write a Hindi children's story
about kindness
for children aged 5-7.
"""

print(generate_story(prompt))

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


एक दिन, एक बड़ी लड़की के नाम जेम्स था। उसने अपने पास एक छोटा सा खेल चलाया था। उसने एक बड़ी गेंद और एक बड़ी टैक्सी खेलने के लिए खेला। उसने अपने पास एक बड़ा खेल चलाया था। उसने एक बड़ी गेंद और एक बड़ी टैक्सी खेलने के लिए खेला। उसने अपने पास एक बड़ा खेल चला�


In [99]:
prompt = """
Write a Hindi children's story
about honesty
for children aged 3-5.
"""

print(generate_story(prompt))

एक बार एक छोटा सा दुनिया था। वह महसूस करते हुए खेलती थी। उसके पास एक बड़ा और अच्छा दुनिया था। उसके पास एक बड़ा और अच्छा दुनिया थी। उसके पास एक बड़ा और अच्छा दुनिया थी। उसके पास एक बड़ा और अच्छा दुनिया थी। उसके पास एक बड़ा और अच्छा दुनिया थी। उसके पास एक बड़ा �


In [100]:
# Loading the base model

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [101]:
def generate_base(prompt):

    messages = [
        {
            "role":"user",
            "content":prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(base_model.device)

    outputs = base_model.generate(
        **inputs,
        max_new_tokens=350
    )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

In [102]:
# Comparision

prompt = "Write a Hindi children's story about honesty."

print("BASE MODEL\n")
print(generate_base(prompt))

print("\n"*3)

print("FINE-TUNED MODEL\n")
print(generate_story(prompt))

BASE MODEL

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Write a Hindi children's story about honesty.
assistant
Once upon a time in the village of Vasantpur, there was a young boy named Ruchi who loved to tell lies. He would often say things that weren't true just because he didn't want to face any consequences.
One day, Ruchi went to visit his grandmother and asked her if she had any sweets for him. His grandmother said no, but Ruchi lied and told her that she did have some delicious sweets for him.
When Ruchi returned home, his mother found out what he had done. She scolded him for lying to her and promised that he would never be allowed to lie again.
Ruchi felt very sad and upset at first, but then he remembered something important. He realized that telling the truth could help people avoid getting hurt or causing harm to others.
So, Ruchi decided to become honest from now on. From that day forward, he always told the truth when it was asked of h

In [77]:
# ROUGE Evaluation

!pip install rouge-score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=1772a6a6f26b86d17d97265b1946a186446d98bae15a4fd0a3eb48fc21469e0f
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


In [90]:
# !pip install evaluate

In [103]:
from evaluate import load

rouge = load("rouge")

In [104]:
predictions = [
    generate_story(
        "Write a Hindi children's story about kindness."
    )
]

references = [
    hindi[0]
]

results = rouge.compute(
    predictions=predictions,
    references=references
)

print(results)

{'rouge1': np.float64(0.0), 'rouge2': np.float64(0.0), 'rougeL': np.float64(0.0), 'rougeLsum': np.float64(0.0)}


In [89]:
# !pip install bert_score

In [84]:
# Bert Score

bertscore = load("bertscore")

In [88]:
# results = bertscore.compute(
#     predictions=predictions,
#     references=references,
#     lang="hi"
# )

# print(results)

In [105]:
dataset[0]

{'messages': [{'role': 'user',
   'content': "Write a Hindi children's story for children aged 7-9 about Honesty."},
  {'role': 'assistant',
   'content': 'एक दिन लिली नाम की एक छोटी लड़की ने अपने कमरे में एक सुई पाई। वह जानती थी कि इसके साथ खेलना मुश्किल है क्योंकि यह तेज था। लिली अपनी मां के साथ सुई साझा करना चाहती थी, इसलिए वह अपनी शर्ट पर एक बटन सीढ़ी सकती थी। लिली अपनी मां के पास गई और बोली, "माँ, मैंने यह सुई पाई है। क्या आप इसे मेरे साथ साझा कर सकते हैं और मेरी शर्ट सीढ़ी सकते हैं?" उसकी मां मुस्कुराई और बोली, "हाँ, लिली, हम सुई साझा कर सकते हैं और अपनी शर्ट ठीक कर सकते हैं। " उन्होंने एक साथ सुई साझा की और लिली की शर्ट पर बटन सीढ़ी। उनके लिए यह मुश्किल नहीं था क्योंकि वे साझा कर रहे थे और एक-दूसरे की मदद कर रहे थे। जब वे खत्म कर चुके थे, तो लिली ने अपनी मां को सुई साझा करने और अपनी शर्ट को ठीक करने के लिए धन्यवाद दिया। वे दोनों खुश महसूस करते थे क्योंकि वे साझा कर रहे थे और साथ काम कर रहे थे।'}],
 'text': '<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a he

In [94]:
prompts = []
references = []

for sample in dataset:

    prompt = sample["messages"][0]["content"]
    response = sample["messages"][1]["content"]

    prompts.append(prompt)
    references.append(response)

print(prompts[0])
print()
print(references[0][:200])

Write a Hindi children's story for children aged 7-9 about Honesty.

एक दिन लिली नाम की एक छोटी लड़की ने अपने कमरे में एक सुई पाई। वह जानती थी कि इसके साथ खेलना मुश्किल है क्योंकि यह तेज था। लिली अपनी मां के साथ सुई साझा करना चाहती थी, इसलिए वह अपनी शर्ट पर एक बटन सीढ़


In [108]:
import time

start = time.time()

_ = generate_story(prompts[0])

print(time.time()-start)

17.409008741378784


In [111]:
from tqdm import tqdm

predictions = []
references = []

NUM_SAMPLES = 20   # Start with 20; increase later if needed

for sample in tqdm(dataset.select(range(NUM_SAMPLES))):

    prompt = sample["messages"][0]["content"]

    reference = sample["messages"][1]["content"]

    prediction = generate_story(prompt)

    predictions.append(prediction.strip())

    references.append(reference.strip())

100%|██████████| 20/20 [05:44<00:00, 17.24s/it]


In [112]:
# ROUGE computation

results = rouge.compute(
    predictions=predictions,
    references=references,
    use_stemmer=False
)

print("ROUGE Scores")
print("--------------------")
for k, v in results.items():
    print(f"{k}: {v:.4f}")

ROUGE Scores
--------------------
rouge1: 0.0000
rouge2: 0.0000
rougeL: 0.0000
rougeLsum: 0.0000


In [113]:
print("PROMPT:\n")
print(dataset[0]["messages"][0]["content"])

print("\nREFERENCE:\n")
print(references[0])

print("\nPREDICTION:\n")
print(predictions[0])

PROMPT:

Write a Hindi children's story for children aged 7-9 about Honesty.

REFERENCE:

एक दिन लिली नाम की एक छोटी लड़की ने अपने कमरे में एक सुई पाई। वह जानती थी कि इसके साथ खेलना मुश्किल है क्योंकि यह तेज था। लिली अपनी मां के साथ सुई साझा करना चाहती थी, इसलिए वह अपनी शर्ट पर एक बटन सीढ़ी सकती थी। लिली अपनी मां के पास गई और बोली, "माँ, मैंने यह सुई पाई है। क्या आप इसे मेरे साथ साझा कर सकते हैं और मेरी शर्ट सीढ़ी सकते हैं?" उसकी मां मुस्कुराई और बोली, "हाँ, लिली, हम सुई साझा कर सकते हैं और अपनी शर्ट ठीक कर सकते हैं। " उन्होंने एक साथ सुई साझा की और लिली की शर्ट पर बटन सीढ़ी। उनके लिए यह मुश्किल नहीं था क्योंकि वे साझा कर रहे थे और एक-दूसरे की मदद कर रहे थे। जब वे खत्म कर चुके थे, तो लिली ने अपनी मां को सुई साझा करने और अपनी शर्ट को ठीक करने के लिए धन्यवाद दिया। वे दोनों खुश महसूस करते थे क्योंकि वे साझा कर रहे थे और साथ काम कर रहे थे।

PREDICTION:

एक दिन, एक बड़ी लड़की के नाम जेम्स था। उसने अपने पास एक बड़ा खेल खेलना चाहता था। उसने अपने पास एक बड़ा खेल खेलना चाहता था। उसने अपने पास ए

In [114]:
print(predictions[0])
print("-" * 80)
print(references[0])

एक दिन, एक बड़ी लड़की के नाम जेम्स था। उसने अपने पास एक बड़ा खेल खेलना चाहता था। उसने अपने पास एक बड़ा खेल खेलना चाहता था। उसने अपने पास एक बड़ा खेल खेलना चाहता था। उसने अपने पास एक बड़ा खेल खेलना चाहता था। उसने अपने पास एक बड़ा खेल खेलना चाहता था। उसने अ
--------------------------------------------------------------------------------
एक दिन लिली नाम की एक छोटी लड़की ने अपने कमरे में एक सुई पाई। वह जानती थी कि इसके साथ खेलना मुश्किल है क्योंकि यह तेज था। लिली अपनी मां के साथ सुई साझा करना चाहती थी, इसलिए वह अपनी शर्ट पर एक बटन सीढ़ी सकती थी। लिली अपनी मां के पास गई और बोली, "माँ, मैंने यह सुई पाई है। क्या आप इसे मेरे साथ साझा कर सकते हैं और मेरी शर्ट सीढ़ी सकते हैं?" उसकी मां मुस्कुराई और बोली, "हाँ, लिली, हम सुई साझा कर सकते हैं और अपनी शर्ट ठीक कर सकते हैं। " उन्होंने एक साथ सुई साझा की और लिली की शर्ट पर बटन सीढ़ी। उनके लिए यह मुश्किल नहीं था क्योंकि वे साझा कर रहे थे और एक-दूसरे की मदद कर रहे थे। जब वे खत्म कर चुके थे, तो लिली ने अपनी मां को सुई साझा करने और अपनी शर्ट को ठीक करने क

In [115]:
import torch

def generate_story(prompt):

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():

        outputs = model.generate(
            **inputs,
            max_new_tokens=400,          # Increase
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
            use_cache=True,
            return_dict_in_generate=True,
        )

    generated = tokenizer.decode(
        outputs.sequences[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    )

    print("Generated tokens:",
          outputs.sequences.shape[1]-inputs.input_ids.shape[1])

    return generated

In [116]:
prompt = """
Write a Hindi children's story
about honesty
for children aged 3-5.
"""

print(generate_story(prompt))

Generated tokens: 400
एक बार के लिए, एक छोटा सा दुनिया में एक नया खेल था। उसकी शरीर पर एक अच्छा गैलरी है। इस घड़ी में एक बहुत बड़ा और बहुत ज्ञात व्यक्ति था। उसका नाम थाका था। उसके बाहर एक बड़ा चाँद था। उसके बाहर एक बड़ा चाँद था। उसके बाहर एक बड़ा चाँद था। उसके बाहर एक बड़ा चाँद था। उसके बाहर एक बड़ा चाँद था। उसके बाहर एक बड़ा चाँद था। उसके बाहर एक बड़ा चाँद था। उसके बाहर एक बड़ा चाँद था। उसके बाहर एक बड़ा चाँद थ


In [117]:
print(len(tokenizer.encode(references[0])))
print(len(tokenizer.encode(predictions[0])))

687
256


In [119]:
prompt = """
Write a Hindi children's story
about honesty
for children aged 3-5.
"""

print(generate_base(prompt))

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user

Write a Hindi children's story
about honesty
for children aged 3-5.

assistant
Once upon a time in the land of Mathura, there lived a little boy named Raju. Raju was very naughty and always lied to his friends and family. He would steal cookies from the cookie jar and then say he didn't do it.

One day, Raju went to visit his grandmother at her house. His grandmother told him that she had baked a big batch of cookies for all their neighbors. Raju thought this was a good idea because he wanted to share with everyone.

When Raju came back home, he said to his grandmother, "I ate some of those yummy cookies you made!" But when his grandmother asked if he had eaten any of them, Raju started crying because he knew he had lied.

Raju's grandmother forgave him and told him that it wasn't important what happened as long as Raju learned from his mistake. She gave Raju a new cookie jar and encouraged him to be hone

In [120]:
prompt = """
३-५ वर्ष के बच्चों के लिए
ईमानदारी पर
एक सरल हिन्दी कहानी लिखिए।
"""

print(generate_story(prompt))

Generated tokens: 400
एक अनुभवी और आसान इतिहास का श्रेणी था। यह एक छोटी टीम थी, जिसमें मेरा भाई था। उन्होंने अपने दोस्तों के साथ एक गहरा घर खोजा। उन्होंने अपने दोस्तों के साथ एक बड़ा ड्रामा खेलने के लिए एक बड़ा धार्मिक टीम बनाया। उन्होंने अपने दोस्तों के साथ एक बड़ा ड्रामा खेलने के लिए एक बड़ा धार्मिक टीम बनाया। उन्होंने अपने दोस्तों के साथ एक बड़ा ड्रामा खेलने के लिए एक बड़ा धार्मिक टीम बनाया। उन्होंने अपने दोस्तों के साथ एक बड़ा


In [122]:
# !pip install gradio

In [123]:
import gradio as gr

themes = [
    "Honesty",
    "Friendship",
    "Kindness",
    "Teamwork",
    "Courage"
]

ages = [
    "3-5",
    "5-7",
    "7-9"
]


def inference(theme, age):

    prompt = f"""
    Write a Hindi children's story
    about {theme}
    for children aged {age}.
    """

    return generate_story(prompt)


demo = gr.Interface(

    fn=inference,

    inputs=[
        gr.Dropdown(themes, label="Theme"),
        gr.Dropdown(ages, label="Age Group")
    ],

    outputs="text",

    title="BalKatha",

    description="Hindi Children's Story Generator powered by Qwen2.5 + LoRA"

)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://bba464c66a647ff95d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
